In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.metrics import recall_score,precision_score,f1_score

In [2]:
df = pd.read_csv('../../../../DataBox/heart_disease.csv')

df = df.drop(columns=['id', 'dataset'])
df = df.rename(columns={'num': 'target'})

df = df.dropna()

In [3]:
df['target'] = (df['target'] > 0).astype(int)

df['sex'] = df['sex'].map({'Female': 0, 'Male': 1})

df['fbs'] = df['fbs'].astype(float).astype(int) 
df['exang'] = df['exang'].astype(int)

cp_order = ['atypical angina', 'non-anginal', 'typical angina', 'asymptomatic']
restecg_order = ['normal', 'st-t abnormality', 'lv hypertrophy']
slope_order = ['upsloping', 'flat', 'downsloping']
thal_order = ['normal', 'fixed defect', 'reversable defect']

encoder = OrdinalEncoder(categories=[cp_order, restecg_order, slope_order, thal_order])

cols_to_encode = ['cp', 'restecg', 'slope', 'thal']
df[cols_to_encode] = encoder.fit_transform(df[cols_to_encode].fillna('normal'))

df = df.reset_index(drop=True)

In [4]:
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,0:-1],df.iloc[:,-1],test_size=0.2,random_state=2)

clf1 = LogisticRegression()
clf2 = DecisionTreeClassifier()

clf1.fit(X_train,y_train)
clf2.fit(X_train,y_train)

DecisionTreeClassifier()

y_pred1 = clf1.predict(X_test)
y_pred2 = clf2.predict(X_test)

C:\Users\parij\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Accuracy

In [5]:
print("Accuracy of Logistic Regression",accuracy_score(y_test,y_pred1))
print("Accuracy of Decision Trees",accuracy_score(y_test,y_pred2))

Accuracy of Logistic Regression 0.7833333333333333
Accuracy of Decision Trees 0.6666666666666666


Problem With Accuracy Score :-  

It doesnt tell where we did mistake   
      1-> Predicted Yes for No   
      2-> Predicted No for Yes  

Both are crutial for Different type of problems  
Ex -> If predicted no cancer for patient having cancer will be Dangerous -> Possible Loss of Life  

To Solve this Problem we have Confusion metric  

#### Confusion Matrix

In [6]:
print("Logistic Regression Confusion Matrix\n")
print(pd.DataFrame(confusion_matrix(y_test,y_pred1),columns=list(range(0,2))))

print("\nDecision Tree Confusion Matrix\n")
pd.DataFrame(confusion_matrix(y_test,y_pred2),columns=list(range(0,2)))

Logistic Regression Confusion Matrix

    0   1
0  30   6
1   7  17

Decision Tree Confusion Matrix



,0,1
0,23,13
1,7,17


In [7]:
result = pd.DataFrame()
result['Actual Label'] = y_test
result['Logistic Regression Prediction'] = y_pred1
result['Decision Tree Prediction'] = y_pred2

result.sample(10)

,Actual Label,Logistic Regression Prediction,Decision Tree Prediction
134,0,0,0
11,0,0,0
188,0,0,0
181,0,1,1
283,0,0,1
222,0,0,0
156,1,1,1
29,1,1,0
270,1,0,1
3,0,0,1


### Confusion matrix

  -----|----1----|----0----  
  -----|---------|---------  
  --1--|----TP---|---FN----  
  --0--|----FP---|---TN----  


##### Type One Error
Number of FP -> Predicted Yes but Actual No

##### Type Two Error
Number of FN -> Predicted No But Actual Yes

### This Also get Misleading when Data Is Imbalanced

Ex -> out of 100000 Passenger 1 is Terrorist 
   -> This will give Good Accuracy and Type 1 and Type 2 Score but Its Dangerous OutPut

##### So To Solve This Problem we Have Precision and Recall

##### Precision  --> (False Alarms)
Ex -> Bad Product Recomendation
###### -> TP/(TP+FP)

##### Recall   --> (Avoide "Missing Out")
Ex -> Dangerous and High Stake
###### -> TP/(TP+FN)

In [8]:
print("For Logistic regression Model")
print("-"*50)
cdf = pd.DataFrame(confusion_matrix(y_test,y_pred1),columns=list(range(0,2)))
print(cdf)
print("-"*50)
print("Precision - ",precision_score(y_test,y_pred1))
print("Recall - ",recall_score(y_test,y_pred1))

For Logistic regression Model
--------------------------------------------------
    0   1
0  30   6
1   7  17
--------------------------------------------------
Precision -  0.7391304347826086
Recall -  0.7083333333333334


In [9]:
print("For DT Model")
print("-"*50)
cdf = pd.DataFrame(confusion_matrix(y_test,y_pred2),columns=list(range(0,2)))
print(cdf)
print("-"*50)
print("Precision - ",precision_score(y_test,y_pred2))
print("Recall - ",recall_score(y_test,y_pred2))

For DT Model
--------------------------------------------------
    0   1
0  23  13
1   7  17
--------------------------------------------------
Precision -  0.5666666666666667
Recall -  0.7083333333333334


Precision is Quality Metric  
Recall is Safety Metric  
  
    
### Use F1 Score when Dont know which Error Is more Dangerous (Type 1 or Type 2)

Here we Calculate Harmonic mean ->  2PR / P+R
P -> Precision
R -> Recall

-> It Penalize Lower Side more (P or R)

In [10]:
print("F1 score LR - ",f1_score(y_test,y_pred1))


print("F1 score DT - ",f1_score(y_test,y_pred2))

F1 score LR -  0.723404255319149
F1 score DT -  0.6296296296296297
